In [1]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

In [2]:
# Configuração 

dataDir = "data"
nPoints = 150          
imgExts = ("*.jpg", "*.jpeg", "*.png")
nFourierHarmonics = 12  
randomState = 42

In [3]:
def imread_unicode(image_path):
    try:
        data = np.fromfile(image_path, dtype=np.uint8)
    except FileNotFoundError:
        return None
    if data.size == 0:
        return None
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

In [4]:
# Extração (imagem + série temporal univariada)

def extract_shape_signature(image_path, n_points=nPoints, debug_plot=False):
    img = imread_unicode(image_path)
    if img is None:
        raise FileNotFoundError(f'Não foi possivel ler: {image_path}')

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    block_size = max(25, (min(gray.shape) // 8) | 1)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
        block_size, 10
    )

    kernel_open = np.ones((3, 3), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_open)

    kernel_close = np.ones((5, 5), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close)

    contours, hierarchy = cv2.findContours(binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
    if not contours:
        raise ValueError(f'Nenhum contorno encontrado em: {image_path}')
    hierarchy = hierarchy[0]

    h_img, w_img = binary.shape

    def touches_border(cnt, margin=8):
        x, y, w, h = cv2.boundingRect(cnt)
        return x <= margin or y <= margin or (x + w) >= w_img - margin or (y + h) >= h_img - margin

    def depth(idx):
        d = 0
        parent = hierarchy[idx][3]
        while parent != -1:
            d += 1
            parent = hierarchy[parent][3]
        return d
    
    img_area = h_img * w_img
    min_area = max(150.0, img_area * 0.0008)
    sized = [c for c in contours if cv2.contourArea(c) >= min_area]
    if not sized:
        sized = list(contours) 

    idx_by_id = {id(c): i for i, c in enumerate(contours)}

    depths = [depth(idx_by_id[id(c)]) for c in sized]
    max_depth = max(depths)
    candidates = [c for c, d in zip(sized, depths) if d == max_depth]

    non_border = [c for c in candidates if not touches_border(c)]
    if non_border:
        candidates = non_border

    contour = max(candidates, key=cv2.contourArea).squeeze()
    if contour.ndim == 1:
        raise ValueError(f'Contorno inválido em: {image_path}')
    
    # centróide via momentos
    M = cv2.moments(contour)
    if M['m00'] == 0:
        cx, cy = contour[:, 0].mean(), contour[:, 1].mean()
    else:
        cx, cy = M['m10'] / M['m00'], M['m01'] / M['m00']

    xs, ys = contour[:, 0].astype(float), contour[:, 1].astype(float)
    raw_signal = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)
    raw_len = len(raw_signal)

    # reamostragem por comprimento de arco
    arc = np.linspace(0, 1, raw_len)
    f = interp1d(arc, raw_signal, kind="linear")
    target = np.linspace(0, 1, n_points)
    signal = f(target)

    # normalização de escala
    max_r = signal.max()
    if max_r > 0:
        signal = signal / max_r

    if debug_plot:
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax[0].plot(contour[:, 0], contour[:, 1], 'r-', linewidth=1)
        ax[0].plot(cx, cy, 'b+', markersize=12)
        ax[0].set_title('Contorno + centróide')
        ax[1].plot(signal)
        ax[1].set_title('Série: distância ao centróide')
        plt.tight_layout()
        plt.savefig(image_path + '.debug.png')
        plt.close()

    return signal, raw_len

In [5]:
# Carregamento do dataset

def load_dataset(split_dir):
    X, y, paths = [], [], []
    class_names = sorted(
        d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))
    )
    for cls in class_names:
        cls_dir = os.path.join(split_dir, cls)
        files = []
        for ext in imgExts:
            files.extend(glob.glob(os.path.join(cls_dir, ext)))
        for fp in sorted(files):
            try:
                sig, _ = extract_shape_signature(fp)
                X.append(sig)
                y.append(cls)
                paths.append(fp)
            except (FileNotFoundError, ValueError) as e:
                print(f'AVISO --> pulando {fp}: {e}')
    return np.array(X), np.array(y), paths, class_names

In [7]:
# Linha de base: 1-NN com Dynamic Type Warping

def dtw_distance(a, b):
    n, m = len(a), len(b)
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = abs(ai - b[j - 1])
            D[i, j] = cost + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return D[n, m]

def knn1_dtw_predict(X_train, y_train, X_test):
    preds = []
    for xt in X_test:
        dists = [dtw_distance(xt, xr) for xr in X_train]
        preds.append(y_train[int(np.argmin(dists))])
    return np.array(preds)

In [8]:
# Classificador escolhido: descritores de Fourier + Random Forest

def fourier_features(signal, n_harmonics=nFourierHarmonics):
    centered = signal - signal.mean()
    spec = np.abs(np.fft.rfft(centered))
    feat = spec[1 : n_harmonics + 1]
    if len(feat) < n_harmonics:
        feat = np.pad(feat, (0, n_harmonics - len(feat)))

    # normalização para energia total = 1
    norm = np.linalg.norm(feat)
    if norm > 0:
        feat = feat / norm
    return feat

def build_feature_matrix(X_series, n_harmonics=nFourierHarmonics):
    return np.array([fourier_features(s, n_harmonics) for s in X_series])

In [9]:
# Avaliação 

def evaluate(y_true, y_pred, class_names, label):
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    print(f'Acurácia: {acc:.3f}')
    print('Matriz de confusão (linhas=verdadeiro, colunas=predito):')
    print('classes:', class_names)
    print(cm)
    print(classification_report(y_true, y_pred, labels=class_names, zero_division=0))

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Verdadeiro')
    ax.set_title(f'{label} Acurácia = {acc:.3f}')
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
    fig.colorbar(im)
    plt.tight_layout()
    out_path = f'confusion_matrix_{label.replace(' ', '_').replace('-', '_')}.png'
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f'figura salva em {out_path}')
    return acc, cm

In [ ]:
# Main

def main():
    train_dir = os.path.join(dataDir, 'train')
    test_dir = os.path.join(dataDir, 'test')

    print('Carregando conjunto de treino')
    X_train, y_train, _, class_names = load_dataset(train_dir)
    print(f'{len(X_train)} exemplos, classes: {class_names}')
    for c in class_names:
        print(f'{c}: {(y_train == c).sum()} exemplos')

    print('Carregando conjunto de teste')
    X_test, y_test, _, class_names_test = load_dataset(test_dir)
    print(f'{len(X_test)} exemplos')
    for c in class_names_test:
        print(f'{c}: {(y_test == c).sum()} exemplos')

    assert len(X_train) >= 30, 'Mínimo de 30 exemplos de treino não atingido'
    assert len(X_test) >= 30, 'Mínimo de 30 exemplos de teste não atingido'

    print('Rodando baseline 1NN-DTW')
    y_pred_dtw = knn1_dtw_predict(X_train, y_train, X_test)
    evaluate(y_test, y_pred_dtw, class_names, 'Baseline 1NN-DTW')

    print('Treinando classificador escolhido --> Fourier + Random Forest')
    Xf_train = build_feature_matrix(X_train)
    Xf_test = build_feature_matrix(X_test)
    clf = RandomForestClassifier(n_estimators=200, random_state=randomState)
    clf.fit(Xf_train, y_train)
    y_pred_rf = clf.predict(Xf_test)
    evaluate(y_test, y_pred_rf, class_names, 'Fourier + Random Forest')

if __name__ == "__main__":
    main()